# Network Security Scanner

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import socket
import threading
import subprocess
import json
from datetime import datetime
import hashlib
import os

class NetworkScanner:
    def __init__(self, target_range="192.168.1.1-254"):
        self.target_range = target_range
        self.open_ports = {}
        self.vulnerabilities = []
        self.scan_results = {}

    def ping_host(self, host):
        try:
            result = subprocess.run(['ping', '-c', '1', '-W', '1000', host],
                                  capture_output=True, text=True, timeout=2)
            return result.returncode == 0
        except:
            return False

    def scan_port(self, host, port):
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((host, port))
            sock.close()
            return result == 0
        except:
            return False

    def banner_grab(self, host, port):
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(2)
            sock.connect((host, port))
            sock.send(b'HEAD / HTTP/1.0\r\n\r\n')
            banner = sock.recv(1024).decode().strip()
            sock.close()
            return banner
        except:
            return None

    def check_default_creds(self, host, port, service):
        default_creds = {
            'ssh': [('admin', 'admin'), ('root', 'root'), ('admin', 'password')],
            'ftp': [('anonymous', ''), ('admin', 'admin'), ('ftp', 'ftp')],
            'telnet': [('admin', 'admin'), ('root', 'root')]
        }

        if service in default_creds:
            for username, password in default_creds[service]:
                if self.test_login(host, port, username, password, service):
                    return True, username, password
        return False, None, None

    def test_login(self, host, port, username, password, service):
        return False

    def detect_service(self, host, port, banner):
        services = {
            21: 'ftp', 22: 'ssh', 23: 'telnet', 25: 'smtp',
            53: 'dns', 80: 'http', 110: 'pop3', 143: 'imap',
            443: 'https', 993: 'imaps', 995: 'pop3s'
        }

        if port in services:
            return services[port]

        if banner:
            banner_lower = banner.lower()
            if 'http' in banner_lower:
                return 'http'
            elif 'ftp' in banner_lower:
                return 'ftp'
            elif 'ssh' in banner_lower:
                return 'ssh'

        return 'unknown'

    def vulnerability_check(self, host, port, service, banner):
        vuln_signatures = {
            'apache/2.2': 'CVE-2012-0883',
            'openssh_5': 'CVE-2010-4478',
            'vsftpd 2.3.4': 'CVE-2011-2523'
        }

        if banner:
            for sig, cve in vuln_signatures.items():
                if sig in banner.lower():
                    self.vulnerabilities.append({
                        'host': host,
                        'port': port,
                        'service': service,
                        'vulnerability': cve,
                        'banner': banner
                    })

    def scan_host(self, host):
        if not self.ping_host(host):
            return

        common_ports = [21, 22, 23, 25, 53, 80, 110, 135, 139, 143, 443, 993, 995, 8080]
        host_results = {'host': host, 'ports': [], 'vulnerabilities': []}

        for port in common_ports:
            if self.scan_port(host, port):
                banner = self.banner_grab(host, port)
                service = self.detect_service(host, port, banner)

                port_info = {
                    'port': port,
                    'service': service,
                    'banner': banner
                }

                has_default, username, password = self.check_default_creds(host, port, service)
                if has_default:
                    port_info['default_creds'] = {'username': username, 'password': password}

                host_results['ports'].append(port_info)
                self.vulnerability_check(host, port, service, banner)

        if host_results['ports']:
            self.scan_results[host] = host_results

    def generate_target_list(self):
        if '-' in self.target_range:
            base_ip, range_part = self.target_range.rsplit('.', 1)
            start, end = map(int, range_part.split('-'))
            return [f"{base_ip}.{i}" for i in range(start, end + 1)]
        else:
            return [self.target_range]

    def threaded_scan(self, targets):
        threads = []
        for target in targets:
            thread = threading.Thread(target=self.scan_host, args=(target,))
            thread.start()
            threads.append(thread)

        for thread in threads:
            thread.join()

    def run_scan(self):
        print(f"Starting network scan on {self.target_range}")
        targets = self.generate_target_list()

        chunk_size = 50
        for i in range(0, len(targets), chunk_size):
            chunk = targets[i:i + chunk_size]
            self.threaded_scan(chunk)

        self.generate_report()

    def generate_report(self):
        report = {
            'scan_time': datetime.now().isoformat(),
            'target_range': self.target_range,
            'hosts_found': len(self.scan_results),
            'total_vulnerabilities': len(self.vulnerabilities),
            'results': self.scan_results,
            'vulnerabilities': self.vulnerabilities
        }

        with open('scan_report.json', 'w') as f:
            json.dump(report, f, indent=2)

        print(f"\nScan completed. Found {len(self.scan_results)} active hosts")
        print(f"Identified {len(self.vulnerabilities)} potential vulnerabilities")
        print("Report saved to scan_report.json")

class LogAnalyzer:
    def __init__(self, log_file_path):
        self.log_file = log_file_path
        self.suspicious_patterns = [
            'failed login', 'authentication failure', 'invalid user',
            'brute force', 'sql injection', 'xss attack', 'directory traversal'
        ]
        self.ip_attempts = {}
        self.alerts = []

    def parse_log_line(self, line):
        parts = line.strip().split()
        if len(parts) < 4:
            return None

        timestamp = ' '.join(parts[:3])
        ip_addr = None

        for part in parts:
            if self.is_valid_ip(part):
                ip_addr = part
                break

        return {
            'timestamp': timestamp,
            'ip': ip_addr,
            'message': line,
            'suspicious': any(pattern in line.lower() for pattern in self.suspicious_patterns)
        }

    def is_valid_ip(self, ip):
        try:
            parts = ip.split('.')
            return len(parts) == 4 and all(0 <= int(part) <= 255 for part in parts)
        except:
            return False

    def analyze_logs(self):
        if not os.path.exists(self.log_file):
            print(f"Log file {self.log_file} not found")
            return

        with open(self.log_file, 'r') as f:
            for line in f:
                entry = self.parse_log_line(line)
                if not entry:
                    continue

                if entry['ip']:
                    if entry['ip'] not in self.ip_attempts:
                        self.ip_attempts[entry['ip']] = 0
                    self.ip_attempts[entry['ip']] += 1

                if entry['suspicious']:
                    self.alerts.append(entry)

        self.detect_brute_force()
        self.generate_security_report()

    def detect_brute_force(self):
        threshold = 10
        for ip, attempts in self.ip_attempts.items():
            if attempts > threshold:
                self.alerts.append({
                    'type': 'brute_force_detection',
                    'ip': ip,
                    'attempts': attempts,
                    'severity': 'high'
                })

    def generate_security_report(self):
        report = {
            'analysis_time': datetime.now().isoformat(),
            'log_file': self.log_file,
            'total_alerts': len(self.alerts),
            'top_attacking_ips': sorted(self.ip_attempts.items(),
                                      key=lambda x: x[1], reverse=True)[:10],
            'alerts': self.alerts
        }

        with open('security_analysis.json', 'w') as f:
            json.dump(report, f, indent=2)

        print(f"Security analysis completed. Found {len(self.alerts)} security alerts")
        print("Report saved to security_analysis.json")

def main():
    print("Cybersecurity Analysis Tool")
    print("1. Network Scanner")
    print("2. Log Analyzer")

    choice = input("Select option (1 or 2): ")

    if choice == '1':
        target = input("Enter target range (default: 192.168.1.1-254): ") or "192.168.1.1-254"
        scanner = NetworkScanner(target)
        scanner.run_scan()

    elif choice == '2':
        log_path = input("Enter log file path: ")
        analyzer = LogAnalyzer(log_path)
        analyzer.analyze_logs()

    else:
        print("Invalid choice")

if __name__ == "__main__":
    main()